## Overview

After initial data cleaning, standardization, and JSON creation, a single victim can still appear in multiple records. Each record may contain partial or overlapping information related to that victim’s experiences, such as sexual violence, abuses, deaths witnessed, border crossings, and money extorted. The JSON aggregation code addresses this by merging all entries for the same victim into one consolidated record, ensuring a unified view of each individual's data.

---

## Data Flow and Processing Steps

1. **Loading the Input Data**:  
   The script begins by loading a previously generated JSON file (`victims.json`) that holds a list of victim records. Each entry contains victim information along with associated crimes, traffickers, borders crossed, and extortion details.

2. **Grouping by Victim ID**:  
   The code groups all records by `victim_id`. This allows the script to gather all entries related to a single victim, making it possible to aggregate their data into one final representation.

3. **Combining Fields Across Records**:  
   For each victim, multiple fields must be merged. The logic for merging depends on the field type:
   - **Binary Indicators (e.g., "yes", "no", "no data")**:  
     The `combine_binary` function processes multiple indicators from various records into a single, logical result. For example, if any record indicates "yes" for sexual violence experienced, the final result is "yes".
   
   - **Lists and Types (e.g., abuse types, death types)**:  
     The `combine_types` function merges lists of strings and filters out placeholder values (like "no data" or "none"). If, after merging, no meaningful data remains, it defaults to "no data".
   
   - **Trafficker Names**:  
     Trafficker names may still appear in multiple variations. The script splits them by "And," removes duplicates, excludes forbidden terms (such as "eritrean" or "manager"), and filters out placeholder entries. The final result is a cleaned list of known traffickers or "no data" if none are identified.
   
   - **Borders Crossed**:  
     The script collects border locations from all records, removes placeholders, and reduces them to a set of unique entries. If no valid borders remain, it defaults to "no data". Otherwise, it stores them as a list of dictionaries with a "border" key.
   
   - **Money Extorted**:  
     Amounts and currencies mentioned in multiple records are combined. If borders are identified, money values are aligned one-to-one with these borders. If actual numeric amounts and meaningful currency codes are found, they are expanded or truncated to match the number of borders. If no valid data is found, it defaults to "no data."

4. **Handling Placeholders and Forbidden Terms**:  
   The code defines a set of placeholder values (`"no data", "none", "unknown", "not specified", "unknown currency"`) and forbidden terms for traffickers. These are removed to ensure that the final JSON contains only relevant, interpretable information.

5. **Adjusting Implausible Values**:  
   If an age is found to be greater than 100, it is replaced with "not specified" to handle data quality issues. Nationalities and other fields containing only placeholder data remain as "no data" or "Unknown" to maintain consistency.

6. **Final JSON Construction**:  
   After aggregating and cleaning all fields, the script creates a single dictionary per victim with the following structure:
   - **victim_id, age, gender, nationality**: Identifiers and demographic information.
   - **trafficker_name**: A cleaned list of trafficker names.
   - **crimes**: A list of dictionaries summarizing sexual violence experienced/witnessed, deaths witnessed, and abuses. Each crime type is unified across all entries.
   - **borders_crossed**: A standardized representation of border crossings, or "no data" if none found.
   - **money_extorted_amount / money_extorted_currency**: Organized and aligned financial extortion details.

   All victim dictionaries are then appended to a `result` list.

7. **Saving the Final Output**:  
   The script saves the aggregated list of victims into a JSON file (`final.json`) with indentation for readability. This final output provides a fully integrated, deduplicated, and standardized dataset, containing one record per victim with complete and coherent data.

---

## Conclusion

By merging multiple entries per victim, removing placeholders and forbidden terms, and harmonizing disparate fields into a single aggregated record, this code produces a clean, final JSON dataset ready for downstream analysis, reporting, or integration into other systems.

replace all the NaN values in the json to  -> "NaN" by using find and replace before running the code below

In [1]:
import json

PLACEHOLDERS = {"no data", "none", "unknown", "not specified", "unknown currency"}
FORBIDDEN_TRAFFICKER_TERMS = {"eritrean", "manager"}

input_file_path = "victims.json"
output_file_path = "cleaned_data.json"

with open(input_file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Group records by victim_id
grouped = {}
for record in data:
    vid = record.get("victim_id")
    vid_str = str(vid) if vid is not None else "no_id"
    if vid_str not in grouped:
        grouped[vid_str] = []
    grouped[vid_str].append(record)

def combine_binary(fields):
    cleaned = [str(f).strip().lower() for f in fields]
    if "yes" in cleaned:
        return "yes"
    if all(f in ("no data", "none") for f in cleaned):
        return "no data"
    return "no"

def combine_types(fields):
    fields_str = [str(f) for f in fields]
    cleaned = [f.lower().strip() for f in fields_str]
    if all(f in ("no data", "none") for f in cleaned):
        return "no data"
    return [fields[i] for i, f in enumerate(cleaned) if f not in ("no data", "none")]

def is_number(s):
    try:
        float(s.replace(",", ""))  # handle commas in numbers
        return True
    except:
        return False

def contains_abuse(labels):
    if labels == "no data" or labels == "no":
        return False
    if not isinstance(labels, list):
        labels = [labels]
    for lbl in labels:
        if "abuse" in str(lbl).lower():
            return True
    return False

result = []

for victim_id, records in grouped.items():
    # Skip victim_id if it's "NaN"
    if victim_id.lower() == "nan":
        continue

    age = records[0].get("age")
    # If age > 100, set to "not specified"
    if isinstance(age, (int, float)) and age > 100:
        age = "not specified"

    gender = records[0].get("gender")
    nationality = records[0].get("nationality")

    # Combine trafficker_name
    trafficker_names = [r.get("trafficker_name", "Unknown") for r in records]

    sexual_violence_experienced_binary_vals = []
    sexual_violence_experienced_type_vals = []
    sexual_violence_witnessed_binary_vals = []
    sexual_violence_witnessed_type_vals = []
    deaths_witnessed_binary_vals = []
    deaths_witnessed_type_vals = []
    abuse_type_experienced_vals = []
    abuse_labels_experienced_vals = []
    abuse_seen_type_vals = []

    for r in records:
        for c in r.get("crimes", []):
            # Sexual Violence Experienced
            if "sexual_violence_experienced_binary" in c:
                sexual_violence_experienced_binary_vals.append(c["sexual_violence_experienced_binary"])
            if "sexual_violence_experienced_type" in c:
                sexual_violence_experienced_type_vals.append(c["sexual_violence_experienced_type"])

            # Sexual Violence Witnessed
            if "sexual_violence_witnessed_binary" in c:
                sexual_violence_witnessed_binary_vals.append(c["sexual_violence_witnessed_binary"])
            if "sexual_violence_witnessed_type" in c:
                sexual_violence_witnessed_type_vals.append(c["sexual_violence_witnessed_type"])

            # Deaths Witnessed
            if "deaths_witnessed_binary" in c:
                deaths_witnessed_binary_vals.append(c["deaths_witnessed_binary"])
            if "deaths_witnessed_type" in c:
                deaths_witnessed_type_vals.append(c["deaths_witnessed_type"])

            # Abuse
            if "abuse_type_experienced" in c:
                abuse_type_experienced_vals.append(c["abuse_type_experienced"])
            if "abuse_labels_experienced" in c:
                val = c["abuse_labels_experienced"]
                if isinstance(val, list):
                    abuse_labels_experienced_vals.extend(val)
                else:
                    abuse_labels_experienced_vals.append(val)
            if "abuse_seen_type" in c:
                val = c["abuse_seen_type"]
                if isinstance(val, list):
                    abuse_seen_type_vals.extend(val)
                else:
                    abuse_seen_type_vals.append(val)

    sexual_violence_experienced_binary = combine_binary(sexual_violence_experienced_binary_vals)
    sexual_violence_experienced_type_combined = combine_types(sexual_violence_experienced_type_vals)
    if isinstance(sexual_violence_experienced_type_combined, list) and not sexual_violence_experienced_type_combined:
        sexual_violence_experienced_type_combined = "no data"

    sexual_violence_witnessed_binary = combine_binary(sexual_violence_witnessed_binary_vals)
    sexual_violence_witnessed_type_combined = combine_types(sexual_violence_witnessed_type_vals)
    if isinstance(sexual_violence_witnessed_type_combined, list) and not sexual_violence_witnessed_type_combined:
        sexual_violence_witnessed_type_combined = "no data"

    deaths_witnessed_binary = combine_binary(deaths_witnessed_binary_vals)
    deaths_witnessed_type_combined = combine_types(deaths_witnessed_type_vals)
    if isinstance(deaths_witnessed_type_combined, list) and not deaths_witnessed_type_combined:
        deaths_witnessed_type_combined = "no data"

    abuse_type_experienced_combined = combine_binary(abuse_type_experienced_vals)
    abuse_labels_experienced_combined = combine_types(abuse_labels_experienced_vals)
    if isinstance(abuse_labels_experienced_combined, list) and not abuse_labels_experienced_combined:
        abuse_labels_experienced_combined = "no data"

    abuse_seen_type_combined = combine_types(abuse_seen_type_vals)
    if isinstance(abuse_seen_type_combined, list) and not abuse_seen_type_combined:
        abuse_seen_type_combined = "no data"

    borders_crossed_combined = []
    for r in records:
        for b in r.get("borders_crossed", []):
            borders_crossed_combined.append(b.get("border"))

    money_extorted_amount_vals = [r.get("money_extorted_amount") for r in records]
    money_extorted_currency_vals = [r.get("money_extorted_currency") for r in records]

    # Flatten money fields if they are lists of values from multiple records
    # They should be aggregated as before, then handled one-to-one with borders
    def flatten_list_of_lists(lst):
        flat = []
        for item in lst:
            if isinstance(item, list):
                flat.extend(item)
            else:
                flat.append(item)
        return flat

    amount_flat = flatten_list_of_lists(money_extorted_amount_vals)
    currency_flat = flatten_list_of_lists(money_extorted_currency_vals)

    # Deduplicate sets for other fields
    def to_set_if_list(value):
        if isinstance(value, list):
            return list(set(value))
        return value

    sexual_violence_experienced_type_combined = to_set_if_list(sexual_violence_experienced_type_combined)
    deaths_witnessed_type_combined = to_set_if_list(deaths_witnessed_type_combined)
    abuse_labels_experienced_combined = to_set_if_list(abuse_labels_experienced_combined)
    abuse_seen_type_combined = to_set_if_list(abuse_seen_type_combined)

    # Adjust abuse_type_experienced
    if contains_abuse(abuse_labels_experienced_combined) and abuse_type_experienced_combined != "yes":
        abuse_type_experienced_combined = "yes"

    # Clean borders
    placeholders_for_borders = {"unknown", "nan", None}
    if isinstance(borders_crossed_combined, list):
        unique_borders = set(borders_crossed_combined)
        actual_places = {b for b in unique_borders if b and b.lower() not in placeholders_for_borders}
        if actual_places:
            unique_borders = actual_places
        else:
            unique_borders = None

        if unique_borders is None or len(unique_borders) == 0:
            borders_crossed_final = "no data"
            num_borders = 0
        else:
            borders_crossed_final = [{"border": b} for b in unique_borders]
            num_borders = len(borders_crossed_final)
    else:
        borders_crossed_final = "no data"
        num_borders = 0

    # nationality placeholder check
    nationality_placeholder = True
    if nationality is not None and isinstance(nationality, str):
        if nationality.strip().lower() not in PLACEHOLDERS:
            nationality_placeholder = False

    # If rape in sexual_violence_experienced_type, remove placeholders
    if isinstance(sexual_violence_experienced_type_combined, list):
        lower_list = [x.lower() for x in sexual_violence_experienced_type_combined]
        if "rape" in lower_list:
            sexual_violence_experienced_type_combined = [x for x in sexual_violence_experienced_type_combined if x.lower() not in PLACEHOLDERS]

    # Trafficker names handling
    # Split on " And "
    final_traffickers = []
    if trafficker_names != "no data":
        if not isinstance(trafficker_names, list):
            trafficker_names = [trafficker_names]
        trafficker_names = list(set(trafficker_names))
        for t in trafficker_names:
            if " And " in t:
                parts = t.split(" And ")
                for p in parts:
                    p = p.strip()
                    if p:
                        final_traffickers.append(p)
            else:
                final_traffickers.append(t)
        final_traffickers = list(set(final_traffickers))
    else:
        final_traffickers = []

    border_places = set()
    if isinstance(borders_crossed_final, list):
        for d in borders_crossed_final:
            p = d.get("border")
            if p and p.lower() not in placeholders_for_borders:
                border_places.add(p.lower())

    forbidden_places = set()
    if not nationality_placeholder and nationality and nationality.strip():
        forbidden_places.add(nationality.lower())

    forbidden_places = forbidden_places.union(border_places)
    cleaned_traffickers = []
    for t in final_traffickers:
        tl = t.lower()
        if tl in PLACEHOLDERS:
            continue
        if tl in forbidden_places:
            continue
        # Remove Eritrean, Manager
        if any(term in tl for term in FORBIDDEN_TRAFFICKER_TERMS):
            continue
        if tl == "unknown":
            continue
        cleaned_traffickers.append(t)

    if len(cleaned_traffickers) == 0:
        cleaned_traffickers = "no data"

    # Handle money arrays one-to-one with borders
    def has_actual_currency(c):
        return c.lower() not in PLACEHOLDERS

    # Determine if we have any actual data
    numeric_amounts = [a for a in amount_flat if a and is_number(a)]
    actual_currencies = [c for c in currency_flat if c and has_actual_currency(c)]

    if num_borders == 0:
        # No borders to map, return sets or no data as previously
        # If no borders, just treat money as before
        def deduplicate_or_no_data(lst):
            if not lst or all(v.lower() in PLACEHOLDERS for v in lst):
                return "no data"
            return list(set(lst))
        money_extorted_amount_final = deduplicate_or_no_data(amount_flat)
        money_extorted_currency_final = deduplicate_or_no_data(currency_flat)
    else:
        # We have some borders
        if numeric_amounts and actual_currencies:
            # Actual data present
            # Match arrays length with borders
            # If less data than borders, replicate last known
            # If more data than borders, truncate (not specified, but logical)
            def expand_or_truncate(lst, length, fill_val):
                if len(lst) == 0:
                    # No data at all, fill with no data
                    return [fill_val]*length
                if len(lst) < length:
                    # replicate last entry
                    last_val = lst[-1]
                    return lst + [last_val]*(length - len(lst))
                return lst[:length]

            amount_expanded = expand_or_truncate(amount_flat, num_borders, "no data")
            currency_expanded = expand_or_truncate(currency_flat, num_borders, "no data")
            money_extorted_amount_final = amount_expanded
            money_extorted_currency_final = currency_expanded
        else:
            # No actual data, just placeholders
            # Deduplicate placeholders into sets as before
            def placeholders_set(lst):
                if not lst or all(v.lower() in PLACEHOLDERS for v in lst):
                    return "no data"
                return list(set(lst))

            # Because the user previously stated if no good info, treat as sets
            money_extorted_amount_final = placeholders_set(amount_flat)
            money_extorted_currency_final = placeholders_set(currency_flat)

    crimes_list = [
        {
          "sexual_violence_experienced_binary": sexual_violence_experienced_binary,
          "sexual_violence_experienced_type": sexual_violence_experienced_type_combined
        },
        {
          "sexual_violence_witnessed_binary": sexual_violence_witnessed_binary,
          "sexual_violence_witnessed_type": sexual_violence_witnessed_type_combined
        },
        {
          "deaths_witnessed_binary": deaths_witnessed_binary,
          "deaths_witnessed_type": deaths_witnessed_type_combined
        },
        {
          "abuse_type_experienced": abuse_type_experienced_combined,
          "abuse_labels_experienced": abuse_labels_experienced_combined,
          "abuse_seen_type": abuse_seen_type_combined
        }
    ]

    combined_record = {
      "victim_id": victim_id,
      "age": age,
      "gender": gender,
      "nationality": nationality,
      "trafficker_name": cleaned_traffickers,
      "crimes": crimes_list,
      "borders_crossed": borders_crossed_final,
      "money_extorted_amount": money_extorted_amount_final,
      "money_extorted_currency": money_extorted_currency_final
    }

    result.append(combined_record)

with open(output_file_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=4)
